# Spot tables → transcripts.parquet
Converts Molecular Cartography spot tables (`.txt`) + segmentation masks into transcript parquets.

**Inputs:** sample overview CSV (with `spot_table` column) + `.tif` masks  
**Outputs:** `{INPUT_DIR}/{sample_id}/transcripts.parquet`  
**Config:** fill in all four paths in the Configuration cell below

## Output schema
`global_x`, `global_y`, `global_z`, `gene`, `overlaps_nucleus`, `transcript_id`, `cell_id`, `EntityID`

In [ ]:
def txt2parquet(sample_id, segmentation_mask, spots_path, output_dir):
    import pandas as pd
    import numpy as np
    import tifffile
    from pathlib import Path
    import glob

    # Load segmentation mask; each pixel value = cell ID (0 = background)
    mask_np = tifffile.imread(segmentation_mask)
    print('loaded mask')
    mask_bool = mask_np > 0

    # Load spot table; keep only x, y, z, gene columns; drop duplicated spots
    spots = pd.read_csv(spots_path, sep='\t', header=None)
    spots = spots.iloc[:, :4]
    spots.columns = ['global_x', 'global_y', 'global_z', 'gene']
    spots = spots[spots['gene'] != 'Duplicated']
    print('loaded spots')

    # Assign cell ID and nucleus overlap by looking up each transcript's pixel value in the mask
    spots['overlaps_nucleus'] = mask_bool[spots['global_y'].astype(int), spots['global_x'].astype(int)]
    spots['transcript_id'] = np.arange(len(spots))
    spots['cell_id'] = mask_np[spots['global_y'].astype(int), spots['global_x'].astype(int)]
    spots['cell_id'] = spots['cell_id'].replace(0, -1)  # background → -1
    spots['EntityID'] = spots['cell_id'].copy()

    spots.to_parquet(f'{output_dir}/{sample_id}/transcripts.parquet', index=False)
    print(f"Saved Parquet for sample {sample_id}")

Main script for running the function

In [ ]:
from pathlib import Path
import glob
import os

# --- CONFIGURATION: fill in paths before running ---
MC_OVERVIEW_CSV = Path("")       # path to cellposesam_segger_sample_overview.csv
INPUT_DIR = Path("")              # output directory (input_for_segger)
MC_SEGMENTATIONS_DIR = Path("")  # directory containing .tif segmentation masks
SDS_BASE_DIR = Path("")          # local mount point — replaces "$HOME/sds-hd/" prefix in spot_table paths

path_table = pd.read_csv(MC_OVERVIEW_CSV)
output_dir = str(INPUT_DIR)

for spots in path_table.spot_table:
    sample_ID = Path(spots).stem.split('_results')[0]
    beast_spots_path = spots.replace("$HOME/sds-hd/", str(SDS_BASE_DIR) + "/")

    # Find the matching .tif mask for this sample
    pattern = os.path.join(str(MC_SEGMENTATIONS_DIR), f"{sample_ID}*.tif")
    mask = glob.glob(pattern)

    txt2parquet(sample_ID, mask, beast_spots_path, output_dir)
